# Batched endpoint soft-wall entropy/charge v2 — lane B

This disjoint lane owns $N_y=[30, 35, 45, 50, 55]$. Dynamics records charge only; all entropy, Rényi, and intrinsic charge-variance eigensolves run after the final occupied frame is durable.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Locked configuration and runtime controls


In [ ]:
from pathlib import Path

LANE = 'B'
BUNDLE_DRIVE_DIR = Path('/content/drive/MyDrive/final_production_new_designs/10_soft_wall_entropy_charge_batched_v2')
OUTPUT_ROOT = Path('/content/drive/MyDrive/classA_final_production_outputs/soft_wall_entropy_charge_nx20_ny30-60_s100_2ny_raster_v2_batched_endpoint')
LOCAL_BUNDLE_DIR = Path('/content/10_soft_wall_entropy_charge_batched_v2_lane_' + LANE)
SCRATCH_ROOT = Path('/content/soft_wall_endpoint_v2_lane_' + LANE + '_scratch')
REPORT_ONLY = False
BENCHMARK_ONLY = False  # True runs/reuses the Ny=60 gate without production.
MAX_NEW_EXECUTION_BATCHES = None

CONFIG = {'Nx': 20,
 'Ny_values': [30, 35, 40, 45, 50, 55, 60],
 'cycles_rule': '2*Ny',
 'device': 'cuda:0',
 'dtype': 'complex128',
 'execution_batch_size_by_Ny': {'30': 80,
                                '35': 60,
                                '40': 40,
                                '45': 30,
                                '50': 25,
                                '55': 20,
                                '60': 20},
 'lane_Ny_values': {'A': [40, 60], 'B': [30, 35, 45, 50, 55]},
 'observer': {'Ay_values': '0..Ny//2',
              'benchmark_maximum_projected_hours': 4.0,
              'benchmark_projection_samples': 20,
              'bootstrap_replicates': 20000,
              'bootstrap_seed': 2026090306,
              'endpoint_only': True,
              'entropy_orders': [1, 2, 3],
              'fit_min_Ay': 8,
              'fixed_contour_Ay': 'Ny//2',
              'fixed_contour_y0': 0,
              'matrix_batch_candidates': [16, 32, 64, 80, 128],
              'minimum_gpu_headroom_gib': 8,
              'occupation_tolerance': 1e-08,
              'origin_average': 'all_periodic_y0_within_trajectory',
              'wall_window_half_width': 1},
 'protocol': {'DW': True,
              'alpha_1': 1.0,
              'alpha_2': 30.0,
              'domain_wall_interval': [5, 15],
              'dw_truncation': False,
              'filling_frac': 0.5,
              'init_mode': 'default',
              'meas_slab_only': False,
              'n_a': 0.5,
              'nshell': 1,
              'perfect_correction': True,
              'postselect': False,
              'postselect_probability': 0.0,
              'sequence': 'raster_y',
              'state_representation': 'physical_frame',
              'trial_orbitals': 'X',
              'triv_region_local_mode': False},
 'result_shard_size': 5,
 'root_seed': 2026090305,
 'samples_per_Ny': 100,
 'sampling_revision': 'soft_wall_entropy_charge_nx20_ny30-60_s100_2ny_raster_v2_batched_endpoint',
 'segment_cycles': 5}
print(f'Lane {LANE}; Ny={CONFIG["lane_Ny_values"][LANE]}')
print(f'Bundle: {BUNDLE_DRIVE_DIR}')
print(f'Output: {OUTPUT_ROOT}')
print('Endpoint products: S1, S2, S3, kappa1, Fq; fixed y0=0 half-strip contours')


## 2. A100 and complex128 check


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required')
name = torch.cuda.get_device_name(0)
gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
if 'A100' not in name.upper() or gib < 38:
    raise RuntimeError(f'A 40-GB-class A100 is required; found {name}, {gib:.1f} GiB')
probe = torch.zeros(1, dtype=torch.complex128, device='cuda:0')
del probe
print(f'Device: {name}; memory={gib:.1f} GiB; dtype=complex128')


## 3. Stage locally, benchmark, and run/resume

The runner streams outer shard, physical-cycle, and endpoint-$A_y$ progress. Production cannot start unless the saved or fresh Ny=60 benchmark projects 20 trajectories in at most four hours.


In [ ]:
import codecs, json, os, shutil, subprocess, sys

def run_streaming_child(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    assert process.stdout is not None
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    try:
        while True:
            raw = os.read(process.stdout.fileno(), 4096)
            if not raw: break
            sys.stdout.write(decoder.decode(raw)); sys.stdout.flush()
        tail = decoder.decode(b'', final=True)
        if tail: sys.stdout.write(tail); sys.stdout.flush()
    except BaseException:
        process.terminate()
        try: process.wait(timeout=5)
        except subprocess.TimeoutExpired: process.kill(); process.wait()
        raise
    finally: process.stdout.close()
    returncode = process.wait()
    if returncode: raise subprocess.CalledProcessError(returncode, command)

if not BUNDLE_DRIVE_DIR.is_dir(): raise FileNotFoundError(BUNDLE_DRIVE_DIR)
if LOCAL_BUNDLE_DIR.exists(): shutil.rmtree(LOCAL_BUNDLE_DIR)
shutil.copytree(BUNDLE_DRIVE_DIR, LOCAL_BUNDLE_DIR)
config_path = Path('/content/soft_wall_endpoint_v2_lane_' + LANE + '_config.json')
config_path.write_text(json.dumps(CONFIG, indent=2, sort_keys=True) + '\n', encoding='utf-8')
command = [sys.executable, '-u', str(LOCAL_BUNDLE_DIR/'run_campaign.py'), '--config', str(config_path), '--lane', LANE, '--output-root', str(OUTPUT_ROOT), '--scratch-root', str(SCRATCH_ROOT)]
if REPORT_ONLY: command.append('--report-only')
if BENCHMARK_ONLY: command.append('--benchmark-only')
if MAX_NEW_EXECUTION_BATCHES is not None: command += ['--max-new-execution-batches', str(int(MAX_NEW_EXECUTION_BATCHES))]
print('[launch]', ' '.join(command), flush=True)
run_streaming_child(command)
print('[notebook] runner exited successfully', flush=True)


## 4. Release the runtime


In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')
